# Step 1: Load and Aggregate NCDB 2019

**Input:** NCDB 2019 (one row per person involved in a collision)
**Output:** `ncdb2019_collisions.csv` (one row per collision, with features and target `Casualty_Class`)

Code meanings come from the NCDB Data Dictionary:
https://open.canada.ca/data/dataset/1eb9eba7-71d1-4b30-9fb1-30cbdab7e63a/resource/21eb7966-38da-4814-a80e-521bce6c4c27/download/data_dictionary.pdf

## 1. Load the raw data
Loads data from 2019 to 2017. If observations are not enough, draws from later years as well

In [1]:
import pandas as pd
import numpy as np

BASE = "https://open.canada.ca/data/dataset/1eb9eba7-71d1-4b30-9fb1-30cbdab7e63a/resource/"
SOURCES = {
    2019: "91d586f9-fb72-4eac-b1f5-d9b95830c877",
    2018: "bf937c33-6fc2-4010-8036-d91d4e1a6285",
    2017: "1e384c9e-6af2-40b3-9457-719fbce819c0",
    2016: "552280a5-f030-41c5-ac7d-649e42f2cb50",
    2015: "cbee1eb5-c853-47fa-b807-0cee9ecd286d",
    2014: "ea6471b8-2924-419b-9ade-baa8d68441ab",
    2013: "c0ea4b35-353c-40fc-9b2b-cd99b5b58801",
    2012: "cd77a978-f506-406e-afd1-b1e61e77dd9a",
    2011: "3fefd7a0-3c49-4eef-97e0-b5310be0fb07",
    2010: "53018df0-82d2-4c9a-8f83-e8f75b3a4300",
    2009: "21b9ce1c-4a83-41fa-ab02-c45b3747c362",
    2008: "60562270-5ffd-4dd7-a7ea-3461bcf245cb",
    2007: "b0366cbf-1400-4b9b-9bea-9ee7ad468b3d",
    2006: "c3c61c6f-706a-41f4-b0c8-bf1e6aa46a8b",
    2005: "d83abcbf-a61e-4fa0-b59a-dfdb4cae4bf8",
    2004: "da00a00b-3355-47f2-b313-58afa7314b15",
    2003: "213ba39d-02cd-43b5-a442-f5f9b80f9d13",
    2002: "b9bafa34-5d55-4353-81b6-f0941ff33bc0",
    2001: "9f696115-84a6-4425-b679-28f677df7c50",
    2000: "d6de0f0f-fad0-4e53-a62a-7024196203d0",
    1999: "80c765bc-eabe-4cc8-86f9-e464cd47e091",
}

FEATURE_COLS = ["C_RCFG", "C_TRAF", "C_WTHR", "C_RSUR", "C_RALN", "C_CONF",
                "C_HOUR", "C_WDAY", "C_MNTH", "C_VEHS"]
SAFETY_KNOWN = {"01", "02", "09", "10", "11", "12", "13"}


def count_usable(part):
    x = part[["C_YEAR", "C_CASE", *FEATURE_COLS, "P_ISEV", "P_SAFE"]].apply(lambda s: s.str.strip())
    x = x.mask(x.apply(lambda s: s.str.fullmatch(r"[UX]+", na=False)))
    for c in ("C_RCFG", "C_TRAF", "C_CONF", "P_SAFE"):
        x[c] = x[c].str.replace(r"^(\d)$", r"0\1", regex=True)
    x["casualty"] = x["P_ISEV"].isin(["2", "3"])
    x["known_safety"] = x["P_SAFE"].isin(SAFETY_KNOWN)
    x["no_safety"] = x["P_SAFE"].isin(["01", "13"])
    g = x.groupby(["C_YEAR", "C_CASE"])
    a = g.agg({**{c: "first" for c in FEATURE_COLS},
               "casualty": "max", "known_safety": "max", "no_safety": "max"})
    for c in ("C_HOUR", "C_WDAY", "C_MNTH", "C_VEHS"):
        a[c] = pd.to_numeric(a[c], errors="coerce")
    valid = (a["C_RCFG"].isin(["02", "03", "09"]) & a["casualty"]
             & a[FEATURE_COLS].notna().all(axis=1) & a["known_safety"])
    return int(valid.sum())


URLS, frames, usable = {}, [], 0
for year in range(2019, 1998, -1):
    url = BASE + f"{SOURCES[year]}/download/y_{year}_en.xlsx"
    part = pd.read_excel(url, dtype=str)
    part.columns = part.columns.str.strip().str.upper()
    assert part["C_YEAR"].str.strip().eq(str(year)).all(), f"Unexpected year in {year} data"
    year_usable = count_usable(part)
    usable += year_usable
    URLS[year] = url
    frames.append(part)
    print(f"{year}: {year_usable:,} usable intersection collisions; total {usable:,}")
    if year <= 2017 and usable >= 50_000:
        break

assert usable >= 50_000, "Fewer than 50,000 usable intersection collisions in 1999–2019"
raw = pd.concat(frames, ignore_index=True)
print(f"Loaded {min(URLS)}–2019; {usable:,} intersection complete cases")
raw.head()

2019: 39,898 usable intersection collisions; total 39,898
2018: 42,876 usable intersection collisions; total 82,774
2017: 44,628 usable intersection collisions; total 127,402
Loaded 2017–2019; 127,402 intersection complete cases


,C_YEAR,C_MNTH,C_WDAY,C_HOUR,C_SEV,C_VEHS,C_CONF,C_RCFG,C_WTHR,C_RSUR,...,V_TYPE,V_YEAR,P_ID,P_SEX,P_AGE,P_PSN,P_ISEV,P_SAFE,P_USER,C_CASE
0,2019,1,1,11,2,2,32,1,1,5,...,1,2015,1,M,56,QQ,1,2,U,2681651
1,2019,1,1,11,2,2,32,1,1,5,...,1,2015,UU,U,UU,UU,U,UU,U,2681651
2,2019,1,1,11,2,2,32,1,1,5,...,1,2010,1,F,30,11,2,2,1,2681651
3,2019,1,1,11,2,2,32,1,1,5,...,1,2010,2,U,1,QQ,N,NN,U,2681651
4,2019,1,1,16,2,2,32,1,2,5,...,1,2007,1,F,27,11,2,2,1,2681652


In [16]:
EXPECTED = ["C_CASE", "C_YEAR", "C_MNTH", "C_WDAY", "C_HOUR", "C_SEV", "C_VEHS",
            "C_CONF", "C_RCFG", "C_WTHR", "C_RSUR", "C_RALN", "C_TRAF",
            "V_ID", "V_TYPE", "V_YEAR", "P_ID", "P_SEX", "P_AGE", "P_PSN",
            "P_ISEV", "P_SAFE", "P_USER"]
            
assert all(set(part.columns) == set(EXPECTED) for part in frames), "Unexpected NCDB columns"
assert raw[["C_YEAR", "C_CASE"]].notna().all().all(), "Missing year or collision ID"
print("Column validation passed for both years")

Column validation passed for both years


## 2. Convert NCDB missing codes

NCDB uses letter codes instead of blanks:
- `U` / `UU` / `UUUU` = unknown, `X` / `XX` = not provided by jurisdiction -> treated as **missing (NaN)**
- `Q` / `QQ` = "other" -> kept as its own category
- `N` / `NN` = not applicable -> kept as its own category

In [17]:
df = raw.copy()
for col in df.columns:
    df[col] = df[col].str.strip()

# Values made only of U and/or X characters are missing
missing_mask = df.apply(lambda s: s.str.fullmatch(r"[UX]+", na=False))

print("Missing codes per column (person level):")
summary = pd.DataFrame({
    "n_missing": missing_mask.sum(),
    "pct_missing": (missing_mask.mean() * 100).round(2),
}).sort_values("pct_missing", ascending=False)
display(summary)

df = df.mask(missing_mask)

# Some years store these codes without the leading zero ("2" instead of "02")
for c in ("C_RCFG", "C_TRAF", "C_CONF", "P_SAFE"):
    df[c] = df[c].str.replace(r"^(\d)$", r"0\1", regex=True)

Missing codes per column (person level):


,n_missing,pct_missing
P_SAFE,94895,11.24
P_AGE,52133,6.17
P_SEX,45759,5.42
C_RCFG,44225,5.24
C_TRAF,40806,4.83
C_RALN,40827,4.83
P_USER,40142,4.75
V_YEAR,37673,4.46
C_CONF,25120,2.97
P_PSN,17275,2.05


## 3. Person-level flags

- `P_ISEV`: 1 = no injury, 2 = injury, 3 = fatality, N = not applicable (e.g. parked car placeholder)
- `P_USER`: 4 = bicyclist; `V_TYPE`: 17 = bicycle
- `V_TYPE` heavy vehicles: 07 unit trucks > 4536 kg, 08 road tractors, 09-11 buses
- `P_SAFE`: 01 = no safety device used, 13 = no device (not available)

In [18]:
HEAVY_TYPES = {"07", "08", "09", "10", "11"}   # edit if you define "heavy" differently
NO_SAFETY = {"01", "13"}
SAFETY_KNOWN = {"01", "02", "09", "10", "11", "12", "13"}

df["is_injury"] = df["P_ISEV"].eq("2")
df["is_fatal"] = df["P_ISEV"].eq("3")
df["isev_unknown"] = df["P_ISEV"].isna()
df["is_cyclist"] = df["P_USER"].eq("4") | df["V_TYPE"].eq("17")
df["is_heavy"] = df["V_TYPE"].isin(HEAVY_TYPES)
df["no_safety"] = df["P_SAFE"].isin(NO_SAFETY)
df["safety_known"] = df["P_SAFE"].isin(SAFETY_KNOWN)

df[["is_injury", "is_fatal", "isev_unknown", "is_cyclist", "is_heavy", "no_safety"]].mean().round(4)

is_injury       0.5221
is_fatal        0.0066
isev_unknown    0.0186
is_cyclist      0.0182
is_heavy        0.0096
no_safety       0.0361
dtype: float64

## 4. Aggregate to one row per collision

In [19]:
COLLISION_COLS = ["C_YEAR", "C_MNTH", "C_WDAY", "C_HOUR", "C_SEV", "C_VEHS",
                  "C_CONF", "C_RCFG", "C_WTHR", "C_RSUR", "C_RALN", "C_TRAF"]

# Sanity check: collision-level fields should be the same for every person in a collision
n_unique = df.groupby("C_CASE")[COLLISION_COLS].nunique()
print("Collisions with inconsistent collision-level values:")
print((n_unique > 1).sum())

Collisions with inconsistent collision-level values:
C_YEAR    0
C_MNTH    0
C_WDAY    0
C_HOUR    0
C_SEV     0
C_VEHS    0
C_CONF    0
C_RCFG    0
C_WTHR    0
C_RSUR    0
C_RALN    0
C_TRAF    0
dtype: int64


In [20]:
agg_spec = {c: (c, "first") for c in COLLISION_COLS if c != "C_YEAR"}
agg_spec.update(
    Num_Persons=("P_ID", "size"),
    Total_Injuries=("is_injury", "sum"),
    Total_Fatalities=("is_fatal", "sum"),
    Unknown_Severity_Persons=("isev_unknown", "sum"),
    Heavy_Vehicle_Present=("is_heavy", "max"),
    Cyclist_Present=("is_cyclist", "max"),
    Any_No_Safety_Device=("no_safety", "max"),
    Safety_Known_Persons=("safety_known", "sum"),
)

col = df.groupby(["C_CASE", "C_YEAR"]).agg(**agg_spec).reset_index()

# Safety device flag: 1 if anyone had no device, 0 if all known people used one,
# NaN if nobody's safety device status is known
col["Safety_Device_Not_Used"] = np.where(
    col["Any_No_Safety_Device"], 1,
    np.where(col["Safety_Known_Persons"] > 0, 0, np.nan))
col = col.drop(columns=["Any_No_Safety_Device", "Safety_Known_Persons"])

for c in ["Heavy_Vehicle_Present", "Cyclist_Present"]:
    col[c] = col[c].astype(int)

print("Collisions:", len(col))
col.head()

Collisions: 330376


,C_CASE,C_YEAR,C_MNTH,C_WDAY,C_HOUR,C_SEV,C_VEHS,C_CONF,C_RCFG,C_WTHR,C_RSUR,C_RALN,C_TRAF,Num_Persons,Total_Injuries,Total_Fatalities,Unknown_Severity_Persons,Heavy_Vehicle_Present,Cyclist_Present,Safety_Device_Not_Used
0,2455891,2017,NaN,NaN,NaN,2,NaN,NaN,01,NaN,NaN,NaN,NaN,20,2,0,0,0,0,NaN
1,2455892,2017,9,1,13,2,1,QQ,NaN,1,6,4,18,1,1,0,0,0,0,0.0
2,2455893,2017,5,5,23,2,1,QQ,02,1,1,3,18,2,1,0,0,0,0,1.0
3,2455894,2017,11,1,9,2,3,NaN,02,3,2,1,NaN,5,2,0,0,0,0,0.0
4,2455895,2017,3,7,8,2,1,04,02,5,3,2,18,2,2,0,0,0,0,0.0


## 5. Derived outcome variables and type conversion

In [21]:
col["Total_Casualties"] = col["Total_Injuries"] + col["Total_Fatalities"]
col["Fatal_Collision"] = (col["Total_Fatalities"] > 0).astype(int)

col["Casualty_Class"] = pd.cut(col["Total_Casualties"], bins=[0, 1, 2, np.inf],
                               labels=["1", "2", "3+"], right=True)
# Collisions with 0 known casualties get NaN (should be rare: NCDB only has injury/fatal collisions)

for c in ["C_YEAR", "C_MNTH", "C_WDAY", "C_HOUR", "C_VEHS"]:
    col[c] = pd.to_numeric(col[c], errors="coerce")

# Intersection flag (C_RCFG 02 = public road intersection, 03 = driveway/parking lot entrance,
# 09 = traffic circle). Kept as a flag so you can decide later whether to filter.
col["Is_Intersection"] = col["C_RCFG"].isin(["02", "03", "09"]).astype(int)

col.dtypes

C_CASE                           str
C_YEAR                         int64
C_MNTH                       float64
C_WDAY                       float64
C_HOUR                       float64
C_SEV                            str
C_VEHS                       float64
C_CONF                           str
C_RCFG                           str
C_WTHR                           str
C_RSUR                           str
C_RALN                           str
C_TRAF                           str
Num_Persons                    int64
Total_Injuries                 int64
Total_Fatalities               int64
Unknown_Severity_Persons       int64
Heavy_Vehicle_Present          int64
Cyclist_Present                int64
Safety_Device_Not_Used       float64
Total_Casualties               int64
Fatal_Collision                int64
Casualty_Class              category
Is_Intersection                int64
dtype: object

## 6. Sanity checks

In [22]:
print("Collisions by year:\n", col.groupby("C_YEAR").size())
assert set(col["C_YEAR"]) == set(URLS)
assert not col[["C_YEAR", "C_CASE"]].duplicated().any()

print("Casualty_Class distribution (%):")
print((col["Casualty_Class"].value_counts(normalize=True, dropna=False) * 100).round(2))

print("\nZero-casualty collisions:", (col["Total_Casualties"] == 0).sum())
print("Collisions with >=1 person of unknown severity:", (col["Unknown_Severity_Persons"] > 0).sum())

# C_SEV: 1 = fatal collision, 2 = non-fatal injury collision
check = pd.crosstab(col["C_SEV"], col["Fatal_Collision"])
print("\nC_SEV vs our Fatal_Collision (should line up):")
print(check)

print("\nIntersection collisions:", col["Is_Intersection"].sum(),
      f"({col['Is_Intersection'].mean() * 100:.1f}%)")
# Final check: match the EDA definition of complete cases among intersections.
EDA_FEATURES = ["C_RCFG", "C_TRAF", "C_WTHR", "C_RSUR", "C_RALN", "C_CONF",
                "C_HOUR", "C_WDAY", "C_MNTH", "C_VEHS", "Heavy_Vehicle_Present",
                "Cyclist_Present", "Safety_Device_Not_Used"]
n_clean = len(col.loc[col["Is_Intersection"].eq(1)].dropna(subset=EDA_FEATURES + ["Casualty_Class"]))
print(f"Usable intersection collisions after aggregation and cleaning: {n_clean:,}")
assert n_clean == usable, f"Loading check ({usable:,}) != aggregated check ({n_clean:,})"
assert n_clean >= 50_000, "Insufficient intersection complete cases"

Collisions by year:
 C_YEAR
2017    114412
2018    111334
2019    104630
dtype: int64
Casualty_Class distribution (%):
Casualty_Class
1      75.77
2      17.30
3+      6.93
NaN     0.00
Name: proportion, dtype: float64

Zero-casualty collisions: 1
Collisions with >=1 person of unknown severity: 9072

C_SEV vs our Fatal_Collision (should line up):
Fatal_Collision       0     1
C_SEV                        
1                     0  5072
2                325304     0

Intersection collisions: 150267 (45.5%)
Usable intersection collisions after aggregation and cleaning: 127,402


In [23]:
print("Missing values per column (collision level, %):")
(col.isna().mean() * 100).round(2).sort_values(ascending=False)

Missing values per column (collision level, %):


Safety_Device_Not_Used      8.31
C_RCFG                      5.53
C_TRAF                      5.37
C_RALN                      4.83
C_CONF                      3.18
C_WTHR                      1.36
C_RSUR                      0.98
C_HOUR                      0.80
C_VEHS                      0.00
C_SEV                       0.00
C_MNTH                      0.00
C_WDAY                      0.00
C_YEAR                      0.00
C_CASE                      0.00
Total_Injuries              0.00
Num_Persons                 0.00
Total_Fatalities            0.00
Unknown_Severity_Persons    0.00
Heavy_Vehicle_Present       0.00
Cyclist_Present             0.00
Total_Casualties            0.00
Fatal_Collision             0.00
Casualty_Class              0.00
Is_Intersection             0.00
dtype: float64

## 7. Save

In [25]:
years = sorted(col["C_YEAR"].dropna().astype(int).unique())
OUT = f"ncdb_{'_'.join(map(str, years))}_collisions.csv"
col.to_csv(OUT, index=False)
print(f"Saved {OUT}: {len(col):,} collisions from {', '.join(map(str, years))}")

# Download automatically when running in Google Colab
try:
    from google.colab import files
    files.download(OUT)
except ImportError:
    pass

Saved ncdb_2017_2018_2019_collisions.csv: 330,376 collisions from 2017, 2018, 2019
